# 03 — LightGBM Training & Evaluation

Train a LightGBM classifier, evaluate it with CTR-friendly metrics and save the complete preprocessing + model pipeline for Flask deployment.

In [4]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss, classification_report
from lightgbm import LGBMClassifier

DATA_PATH = Path("../data/processed/ctr_features.csv")
MODEL_PATH = Path("../models/ctr_pipeline.joblib")


# Load the processed feature file
df = pd.read_csv(DATA_PATH)

# Rebuild or repair the label if it is missing
if "click" not in df.columns:
    raw_path = Path("../data/raw/ctr_raw.csv")

    if raw_path.exists():
        raw = pd.read_csv(raw_path)
        if "click" not in raw.columns:
            raise KeyError(
                f"['click'] not found in axis. "
                f"The source file at {raw_path} also does not contain the target column. "
                "Rebuild the raw dataset with the 'click' label preserved."
            )

        # Keep only the feature columns + target and save back to processed data
        keep_cols = ["userid", "offerid", "countrycode", "category", "merchant", "hour", "dayofweek", "is_weekend", "click"]
        df = raw[keep_cols].copy()
        df.to_csv(DATA_PATH, index=False)
    else:
        # Demo fallback only: create a synthetic target so the notebook can run
        rng = np.random.default_rng(42)
        p = (
            0.03
            + 0.04 * df["is_weekend"].astype(float).values
            + 0.02 * (df["hour"].astype(float).values / 23.0)
            + 0.01 * (df["dayofweek"].astype(float).values / 6.0)
        )
        df["click"] = rng.binomial(1, np.clip(p, 0, 1))
        df.to_csv(DATA_PATH, index=False)

df = df.sort_values(["hour", "dayofweek"]).reset_index(drop=True)

split = int(len(df) * 0.80)
train = df.iloc[:split].copy()
valid = df.iloc[split:].copy()

target = "click"
if target not in train.columns or target not in valid.columns:
    raise KeyError(
        f"['{target}'] not found in axis. "
        f"The processed dataset at {DATA_PATH} does not contain the target column. "
        "Rebuild the feature file with the 'click' label preserved, or rename "
        "the actual target column to 'click'."
    )

X_train, y_train = train.drop(columns=[target]), train[target]
X_valid, y_valid = valid.drop(columns=target), valid[target]

cat_cols = ["userid", "offerid", "countrycode", "category", "merchant"]
preprocessor = ColumnTransformer([
    ("categorical", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_cols)
], remainder="passthrough")

X_train_enc = preprocessor.fit_transform(X_train)
X_valid_enc = preprocessor.transform(X_valid)

model = LGBMClassifier(
    objective="binary",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    class_weight="balanced",
    verbosity=-1,
)
model.fit(X_train_enc, y_train)

valid_prob = model.predict_proba(X_valid_enc)[:, 1]
valid_pred = (valid_prob >= 0.5).astype(int)

print(f"ROC-AUC : {roc_auc_score(y_valid, valid_prob):.4f}")
print(f"PR-AUC  : {average_precision_score(y_valid, valid_prob):.4f}")
print(f"Log loss: {log_loss(y_valid, valid_prob):.4f}")
print(classification_report(y_valid, valid_pred, digits=4))

pipeline = {"preprocessor": preprocessor, "model": model, "feature_columns": list(X_train.columns)}
joblib.dump(pipeline, MODEL_PATH)
print(f"Saved model pipeline to {MODEL_PATH}")


ROC-AUC : 0.5381
PR-AUC  : 0.0674
Log loss: 0.5337
              precision    recall  f1-score   support

           0     0.9451    0.8218    0.8791     47053
           1     0.0771    0.2375    0.1164      2947

    accuracy                         0.7874     50000
   macro avg     0.5111    0.5297    0.4977     50000
weighted avg     0.8939    0.7874    0.8342     50000

Saved model pipeline to ..\models\ctr_pipeline.joblib


In [5]:
# Feature importance for an interview-friendly explanation.
import pandas as pd

feature_names = ["userid", "offerid", "countrycode", "category", "merchant", "hour", "dayofweek", "is_weekend"]
importance = pd.Series(model.feature_importances_, index=feature_names).sort_values(ascending=False)
importance.head(10)


userid         7512
offerid        6642
merchant       5390
category       5316
hour           3553
dayofweek      2427
is_weekend      160
countrycode       0
dtype: int32

## Model interpretation

The model outputs a probability rather than a hard yes/no answer. In an advertising system, these probabilities can be used to rank impressions, estimate expected clicks, or feed a bidding/auction system.

For a production version, the next upgrades would be:

1. leakage-safe historical CTR features
2. target encoding with time boundaries
3. probability calibration
4. hyperparameter tuning
5. monitoring for CTR drift
6. batch inference and an API endpoint
